# 08 - Semantic Inference over the HPO Knowledge Graph

## Learning objectives
- Understand how the HPO phenotype hierarchy enables **semantic inference**.
- Use the Neosemantics procedure `n10s.inference.nodesInCategory` to find diseases
  *implicitly* linked to a phenotype category through the `SUBCLASSOF` hierarchy.
- Appreciate the difference between **direct** and **inferred** annotations.
- Run clinically motivated queries that exploit hierarchical reasoning.

In [1]:
from neo4j import GraphDatabase

In [2]:
# Connection settings.
uri = "bolt://localhost:7687"
user = "neo4j"
password = "12345678"
database = "hpo"

driver = GraphDatabase.driver(uri, auth=(user, password))
driver.verify_connectivity()
print(f"Connected to Neo4j at {uri}")

Connected to Neo4j at bolt://localhost:7687


## Background – Why Semantic Inference Matters

The HPO ontology organises phenotypic abnormalities into a **hierarchy** via
`SUBCLASSOF` relationships.  For example:

```
Growth abnormality
  └── SUBCLASSOF ── Growth delay
  └── SUBCLASSOF ── Abnormality of body height  ──  └── SUBCLASSOF ── Short stature
  └── SUBCLASSOF ── Abnormality of body weight
  └── SUBCLASSOF ── Asymmetric growth
  └── ...
```

A disease may be annotated with a **specific** phenotype such as *Growth delay*,
but **not** directly linked to the broader category *Growth abnormality*.

Using hierarchical inference we can deduce that:

> **If DiseaseX `HAS_PHENOTYPIC_FEATURE` → Growth delay, and Growth delay
> `SUBCLASSOF` → Growth abnormality, then DiseaseX implicitly belongs to the
> category "Growth abnormality".**

This is exactly what the Neosemantics procedure `n10s.inference.nodesInCategory`
does: it traverses the phenotype hierarchy starting at a given category node,
collects all descendant phenotypes reachable via `SUBCLASSOF`, and then returns
every node connected through `HAS_PHENOTYPIC_FEATURE` to any phenotype in that
collected set.

## Step 1 – Direct vs. Inferred Connections (Illustrative Example)

Let us first illustrate the concept with a concrete example.

Consider the category **Growth abnormality**.  We start by finding diseases
that are **directly** linked to this exact node (not to any of its children).

In [3]:
direct_query = """
MATCH (cat:HpoPhenotype {label: "Growth abnormality"})
      <-[:HAS_PHENOTYPIC_FEATURE]-(dis:HpoDisease)
RETURN dis.label AS disease
ORDER BY disease
LIMIT 10
"""

with driver.session(database=database) as session:
    results = session.run(direct_query).data()

print(f"Diseases DIRECTLY linked to 'Growth abnormality': {len(results)}")
for r in results:
    print(f"  - {r['disease']}")

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `HAS_PHENOTYPIC_FEATURE` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=3, column=11, offset=66>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 66, 'line': 3, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (cat:HpoPhenotype {label: "Growth abnormality"})\n      <-[:HAS_PHENOTYPIC_FEATURE]-(dis:HpoDisease)\nRETURN dis.label AS disease\nORDER BY disease\nLIMIT 10\n'


Diseases DIRECTLY linked to 'Growth abnormality': 0


Now let us use `n10s.inference.nodesInCategory` to find diseases **inferred** to
belong to the category *Growth abnormality* — that is, diseases linked to *any*
descendant phenotype (e.g. *Growth delay*, *Abnormality of body height*, …).

In [4]:
inferred_query = """
MATCH (cat:HpoPhenotype {label: "Growth abnormality"})
CALL n10s.inference.nodesInCategory(cat, {
    inCatRel: "HAS_PHENOTYPIC_FEATURE",
    subCatRel: "SUBCLASSOF"
})
YIELD node AS dis
WHERE dis:HpoDisease
RETURN dis.label AS disease
ORDER BY disease
LIMIT 20
"""

with driver.session(database=database) as session:
    results = session.run(inferred_query).data()

print(f"Diseases INFERRED under 'Growth abnormality' (first 20):")
for r in results:
    print(f"  - {r['disease']}")

Diseases INFERRED under 'Growth abnormality' (first 20):


Notice that the inferred set is much larger than the direct one.  Diseases like
those annotated with *Growth delay* now appear even though they are **not**
directly linked to the *Growth abnormality* node.

> **Key insight:** Because *Growth delay* is inside the broader category
> *Growth abnormality* via `SUBCLASSOF`, the system can infer that
> DiseaseX has a phenotype that belongs to the category "Growth abnormality".
> So when you ask "give me diseases in the category Growth abnormality",
> DiseaseX appears **even though it is not directly linked to that category node**.

## Step 2 – Verify the Inference Path for a Specific Disease

To make the mechanism fully transparent, let us pick one disease from the inferred
set and trace the path back through the hierarchy.

In [5]:
trace_query = """
MATCH (cat:HpoPhenotype {label: "Growth abnormality"})
CALL n10s.inference.nodesInCategory(cat, {
    inCatRel: "HAS_PHENOTYPIC_FEATURE",
    subCatRel: "SUBCLASSOF"
})
YIELD node AS dis
WHERE dis:HpoDisease
WITH dis LIMIT 1
MATCH (dis)-[:HAS_PHENOTYPIC_FEATURE]->(phe:HpoPhenotype)
      -[:SUBCLASSOF*1..]->(anc:HpoPhenotype {label: "Growth abnormality"})
RETURN dis.label AS disease,
       collect(DISTINCT phe.label) AS connecting_phenotypes
"""

with driver.session(database=database) as session:
    results = session.run(trace_query).data()

for r in results:
    print(f"Disease: {r['disease']}")
    print(f"  Phenotypes under 'Growth abnormality':")
    for p in r["connecting_phenotypes"]:
        print(f"    - {p}")

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `HAS_PHENOTYPIC_FEATURE` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=10, column=15, offset=240>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 240, 'line': 10, 'column': 15}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (cat:HpoPhenotype {label: "Growth abnormality"})\nCALL n10s.inference.nodesInCategory(cat, {\n    inCatRel: "HAS_PHENOTYPIC_FEATURE",\n    subCatRel: "SUBCLASSOF"\n})\nYIELD node AS dis\nWHERE dis:HpoDisease\nWITH dis LIMIT 1\nMATCH (dis)-[:HAS_PHENOTYPIC_FEATURE]->(phe:HpoPhenotype)\n      -[

## Step 3 – Inferred Annotations for "Abnormality of the Endocrine System"

Using the same hierarchical structure, we can infer annotations implicitly
linked to **Abnormality of the endocrine system** through the Neosemantics
inference procedure.

The query below:
1. Starts at the category node *Abnormality of the endocrine system*.
2. Traverses the phenotype hierarchy via `SUBCLASSOF` to collect all
   descendant phenotypes.
3. Finds all diseases that have `HAS_PHENOTYPIC_FEATURE` pointing to any
   phenotype in that collected set.
4. Filters to a specific set of diseases for readability.
5. For each disease, collects its **full** phenotypic feature list (not just
   the endocrine ones).

In [6]:
endocrine_query = """
MATCH (cat:HpoPhenotype {label: "Abnormality of the endocrine system"})
CALL n10s.inference.nodesInCategory(cat, {
    inCatRel: "HAS_PHENOTYPIC_FEATURE",
    subCatRel: "SUBCLASSOF"
})
YIELD node AS dis
WHERE dis.label IN [
    "Congenital atransferrinemia",
    "Deafness, autosomal recessive 4, with enlarged vestibular aqueduct",
    "Diabetes mellitus, transient neonatal, 1",
    "Edema, familial idiopathic, prepubertal",
    "Familial dysalbuminemic hyperthyroxinemia"
]
MATCH (dis)-[:HAS_PHENOTYPIC_FEATURE]->(phe:HpoPhenotype)
RETURN dis.label AS disease, collect(DISTINCT phe.label) AS features
ORDER BY size(features) ASC, disease
"""

with driver.session(database=database) as session:
    results = session.run(endocrine_query).data()

for r in results:
    print(f"\n{r['disease']}  ({len(r['features'])} features)")
    for f in r["features"]:
        print(f"  - {f}")

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `HAS_PHENOTYPIC_FEATURE` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=15, column=15, offset=494>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 494, 'line': 15, 'column': 15}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (cat:HpoPhenotype {label: "Abnormality of the endocrine system"})\nCALL n10s.inference.nodesInCategory(cat, {\n    inCatRel: "HAS_PHENOTYPIC_FEATURE",\n    subCatRel: "SUBCLASSOF"\n})\nYIELD node AS dis\nWHERE dis.label IN [\n    "Congenital atransferrinemia",\n    "Deafness, autosomal recessi

### Interpretation

The procedure `n10s.inference.nodesInCategory`:

1. **Traverses the phenotype hierarchy** starting at *Abnormality of the endocrine
   system* and collects **all descendant phenotypes** reachable via `SUBCLASSOF`.
2. **Finds all diseases** that have `HAS_PHENOTYPIC_FEATURE` pointing to **any
   phenotype in that collected set**.
3. Returns those disease nodes.

Because the diseases above are annotated with specific endocrine phenotypes
(children or grandchildren of *Abnormality of the endocrine system*), the
inference procedure discovers them even if they have no *direct* edge to the
top-level category.  This is the power of ontology-based semantic inference.

## Step 4 – Count: Direct vs. Inferred

Comparing the two counts makes it clear how much information the hierarchy adds.

In [7]:
comparison_query = """
MATCH (cat:HpoPhenotype {label: "Abnormality of the endocrine system"})
OPTIONAL MATCH (cat)<-[:HAS_PHENOTYPIC_FEATURE]-(d1:HpoDisease)
WITH cat, count(DISTINCT d1) AS directCount
CALL n10s.inference.nodesInCategory(cat, {
    inCatRel: "HAS_PHENOTYPIC_FEATURE",
    subCatRel: "SUBCLASSOF"
})
YIELD node AS d2
WHERE d2:HpoDisease
RETURN directCount,
       count(DISTINCT d2) AS inferredCount
"""

with driver.session(database=database) as session:
    result = session.run(comparison_query).single()
    direct   = result["directCount"]
    inferred = result["inferredCount"]

print(f"Category: Abnormality of the endocrine system")
print(f"  Diseases directly linked:   {direct:,}")
print(f"  Diseases inferred (total):  {inferred:,}")
print(f"  Additional via inference:   {inferred - direct:,}")

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `HAS_PHENOTYPIC_FEATURE` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=3, column=25, offset=97>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 97, 'line': 3, 'column': 25}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (cat:HpoPhenotype {label: "Abnormality of the endocrine system"})\nOPTIONAL MATCH (cat)<-[:HAS_PHENOTYPIC_FEATURE]-(d1:HpoDisease)\nWITH cat, count(DISTINCT d1) AS directCount\nCALL n10s.inference.nodesInCategory(cat, {\n    inCatRel: "HAS_PHENOTYPIC_FEATURE",\n    subCatRel: "SUBCLASSOF"\n})\nYIE

TypeError: 'NoneType' object is not subscriptable

## Step 5 – Explore Other Categories

The same technique works for **any** category in the HPO hierarchy.  Below we
run the inference for several top-level phenotype categories and compare
direct vs. inferred counts.

In [ ]:
categories = [
    "Abnormality of the nervous system",
    "Abnormality of the cardiovascular system",
    "Abnormality of the musculoskeletal system",
    "Abnormality of the endocrine system",
    "Abnormality of the immune system",
    "Growth abnormality",
]

multi_cat_query = """
MATCH (cat:HpoPhenotype {label: $category})
OPTIONAL MATCH (cat)<-[:HAS_PHENOTYPIC_FEATURE]-(d1:HpoDisease)
WITH cat, count(DISTINCT d1) AS directCount
CALL n10s.inference.nodesInCategory(cat, {
    inCatRel: "HAS_PHENOTYPIC_FEATURE",
    subCatRel: "SUBCLASSOF"
})
YIELD node AS d2
WHERE d2:HpoDisease
RETURN $category       AS category,
       directCount,
       count(DISTINCT d2) AS inferredCount
"""

print(f"{'Category':<50} {'Direct':>8} {'Inferred':>10} {'Added':>8}")
print("-" * 80)

with driver.session(database=database) as session:
    for cat in categories:
        result = session.run(multi_cat_query, category=cat).single()
        d = result["directCount"]
        i = result["inferredCount"]
        print(f"{cat:<50} {d:>8,} {i:>10,} {i - d:>8,}")

Category                                             Direct   Inferred    Added
--------------------------------------------------------------------------------
Abnormality of the nervous system                       136      8,451    8,315
Abnormality of the cardiovascular system                 63      4,952    4,889
Abnormality of the musculoskeletal system                 3      8,534    8,531
Abnormality of the endocrine system                      37      1,948    1,911
Abnormality of the immune system                         36      3,410    3,374
Growth abnormality                                       39      4,490    4,451


## Step 6 – A Clinically Motivated Example

Imagine a clinician observes *Growth delay* in a patient. They want to find
diseases in the broader category *Growth abnormality*.

Without semantic inference they would have to manually list every descendant
phenotype of *Growth abnormality* and search for each one individually.
With `n10s.inference.nodesInCategory`, a single query does all the work.

The query below uses the inference procedure to discover diseases and then
collects **all** phenotypic features for each disease so the clinician gets
the full clinical picture.

In [ ]:
clinical_query = """
MATCH (cat:HpoPhenotype {label: "Growth abnormality"})
CALL n10s.inference.nodesInCategory(cat, {
    inCatRel: "HAS_PHENOTYPIC_FEATURE",
    subCatRel: "SUBCLASSOF"
})
YIELD node AS dis
WHERE dis:HpoDisease
WITH dis
MATCH (dis)-[:HAS_PHENOTYPIC_FEATURE]->(phe:HpoPhenotype)
WITH dis.label AS disease, collect(DISTINCT phe.label) AS phenotypes
RETURN disease, phenotypes, size(phenotypes) AS featureCount
ORDER BY featureCount DESC
LIMIT 10
"""

with driver.session(database=database) as session:
    results = session.run(clinical_query).data()

print("Top 10 diseases inferred under 'Growth abnormality':\n")
for r in results:
    print(f"  {r['disease']}  ({r['featureCount']} phenotypic features)")
    for p in r["phenotypes"][:5]:
        print(f"    - {p}")
    if r["featureCount"] > 5:
        print(f"    ... and {r['featureCount'] - 5} more")

Top 10 diseases inferred under 'Growth abnormality':

  Stolerman neurodevelopmental syndrome  (233 phenotypic features)
    - Cleft lip
    - High anterior hairline
    - Hemangioma
    - Bifid uvula
    - Cleft palate
    ... and 228 more
  ReNU syndrome  (211 phenotypic features)
    - Open mouth
    - Bifid uvula
    - Thick lower lip vermilion
    - Narrow palate
    - Wide mouth
    ... and 206 more
  Multiple congenital anomalies-hypotonia-seizures syndrome 2  (202 phenotypic features)
    - Decreased facial expression
    - Increased head circumference
    - Edema
    - High anterior hairline
    - Abnormality of the supraorbital ridges
    ... and 197 more
  Intellectual developmental disorder, X-linked syndromic 37  (201 phenotypic features)
    - High anterior hairline
    - Pierre-Robin sequence
    - Cleft palate
    - Narrow palate
    - Nephrocalcinosis
    ... and 196 more
  Multiple mitochondrial dysfunctions syndrome 9B  (198 phenotypic features)
    - High palate
   

## Summary

| Concept | Explanation |
|---------|-------------|
| **Direct annotation** | A disease is explicitly linked to a phenotype node via `HAS_PHENOTYPIC_FEATURE`. |
| **Inferred annotation** | A disease is linked to a *descendant* of a category via `HAS_PHENOTYPIC_FEATURE`; the inference procedure propagates this to the ancestor category. |
| **`n10s.inference.nodesInCategory`** | Neosemantics procedure that traverses a `subCatRel` hierarchy from a root category and returns all nodes connected via `inCatRel` to any node in the hierarchy. |
| **Clinical benefit** | Clinicians can query at any level of specificity and discover diseases annotated at more specific levels, without manually enumerating subcategories. |

In [ ]:
driver.close()
print("Neo4j driver closed.")
